# 09 Using Execution Log and Resuming from Checkpoints

Notebook generado a partir del paquete Python. Los módulos se incluyen en orden de dependencia (los módulos importados por otros aparecen primero).

> Nota: los `import` de módulos locales del paquete original se conservan tal cual. Como todos los módulos están combinados en este notebook en orden de dependencia, los símbolos referenciados ya quedan definidos en celdas anteriores.

## ¿Qué hace este notebook?

Muestra cómo **inspeccionar el log de ejecución y reanudar desde un checkpoint**. Modela
un flujo simple de análisis de bug: `set_prompt → identify_bug → propose_fix → finalize`,
con `InMemorySaver` guardando estado en cada paso bajo un único `thread_id`.

Las utilidades `state_diff` y `show_timeline` imprimen la **línea de tiempo** de la
ejecución (cada `checkpoint_id`, el próximo nodo y las diferencias de estado). Luego se
puede **reanudar** la ejecución desde cualquier `checkpoint_id` elegido.

## Ejemplo de uso

**Datos de interacción que espera el agente.** El flujo corre solo; el "dato de
continuación" es el `checkpoint_id` desde el que quieres reanudar.

- Entrada inicial: `{}` (el grafo fija el `prompt` internamente).
- Para **continuar** desde un punto pasado, arma `resume_config` con el mismo `thread_id`
  **más** el `checkpoint_id` elegido del timeline, y llama `invoke(None, resume_config)`.

```python
import uuid

graph = build_graph()
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
out = graph.invoke({}, config)                     # ejecución inicial → concluye
print("Salida:", out["output"])

timeline = show_timeline(graph, config)            # lista cada checkpoint_id
cid = timeline[1].config["configurable"]["checkpoint_id"]

resume_config = {"configurable": {
    "thread_id": config["configurable"]["thread_id"],
    "checkpoint_id": cid,                           # dato de interacción para reanudar
}}
out2 = graph.invoke(None, resume_config)           # continúa desde ese checkpoint
print("Reanudado:", out2.get("output"))
```

In [1]:
"""
This graph models a simple bug-analysis workflow: set a bug prompt, identify its cause,
propose a fix, and produce a final output string. LangGraph checkpoints state at each step
under a single thread_id, so we can inspect the full execution history (checkpoint_id, next node,
and state diffs) and resume execution from any chosen checkpoint by passing its checkpoint_id.
"""

'\nThis graph models a simple bug-analysis workflow: set a bug prompt, identify its cause,\npropose a fix, and produce a final output string. LangGraph checkpoints state at each step\nunder a single thread_id, so we can inspect the full execution history (checkpoint_id, next node,\nand state diffs) and resume execution from any chosen checkpoint by passing its checkpoint_id.\n'

In [2]:
import uuid
from typing_extensions import TypedDict, NotRequired
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

In [3]:

# Define State
class State(TypedDict):
    prompt: NotRequired[str]
    bug: NotRequired[str]
    fix: NotRequired[str]
    output: NotRequired[str]

In [4]:

# Nodes
def set_prompt(state: State):
    return {"prompt": "Bug: Python function returns None unexpectedly"}  

In [5]:

def identify_bug(state: State):
    return {"bug": "Missing return statement in a branch"}

In [6]:

def propose_fix(state: State):
    return {"fix": "Add an explicit return value in every branch"}

In [7]:

def finalize(state: State):
    return {
        "output": f"{state['prompt']} | Cause: {state['bug']} | Fix: {state['fix']}"
    }

In [8]:

def build_graph():
    workflow = StateGraph(State)
    workflow.add_node("set_prompt", set_prompt)
    workflow.add_node("identify_bug", identify_bug)
    workflow.add_node("propose_fix", propose_fix)
    workflow.add_node("finalize", finalize)

    workflow.add_edge(START, "set_prompt")
    workflow.add_edge("set_prompt", "identify_bug")
    workflow.add_edge("identify_bug", "propose_fix")
    workflow.add_edge("propose_fix", "finalize")
    workflow.add_edge("finalize", END)

    return workflow.compile(checkpointer=InMemorySaver())

In [9]:

# Helper: Execution log visualization
def state_diff(prev_vals: dict, curr_vals: dict):
    prev_vals = prev_vals or {}
    curr_vals = curr_vals or {}
    changes = {}
    for k in sorted(set(prev_vals) | set(curr_vals)):
        if prev_vals.get(k) != curr_vals.get(k):
            changes[k] = {"from": prev_vals.get(k), "to": curr_vals.get(k)}
    return changes

In [10]:

def show_timeline(graph, config):
    states = list(graph.get_state_history(config))
    states = list(reversed(states))  # chronological

    print("\n=== EXECUTION TIMELINE ===")
    prev = None
    for i, s in enumerate(states):
        cid = s.config["configurable"]["checkpoint_id"]
        nxt = s.next
        vals = dict(s.values or {})
        diff = state_diff(prev.values if prev else {}, vals)
        print(f"\nStep {i}")
        print(" checkpoint_id:", cid)
        print(" next:", nxt)
        print(" state_diff:", diff)
        prev = s

    return states

In [11]:

def checkpoint_exists(timeline, checkpoint_id: str) -> bool:
    return any(
        s.config["configurable"]["checkpoint_id"] == checkpoint_id
        for s in timeline
    )

In [12]:

def main():
    graph = build_graph()

    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    print("\nRUN INITIAL EXECUTION")
    out = graph.invoke({}, config)
    print("Final output:", out["output"])

    timeline = show_timeline(graph, config)

    print("\n" + "-" * 100)
    print("Resume execution from any checkpoint shown above.")
    checkpoint_id = input("checkpoint_id: ").strip()

    if not checkpoint_exists(timeline, checkpoint_id):
        print("\nInvalid checkpoint_id.")
        return

    print("\nRESUMING FROM CHECKPOINT")
    resume_config = {
        "configurable": {
            "thread_id": thread_id,
            "checkpoint_id": checkpoint_id
        }
    }

    out2 = graph.invoke(None, resume_config)

    if "output" in out2:
        print("Final output:", out2["output"])
    else:
        print("Final state:", out2)

    print("\nTimeline after resuming (new checkpoints appended)")
    show_timeline(graph, config)

In [13]:

if __name__ == "__main__":
    main()


RUN INITIAL EXECUTION
Final output: Bug: Python function returns None unexpectedly | Cause: Missing return statement in a branch | Fix: Add an explicit return value in every branch

=== EXECUTION TIMELINE ===

Step 0
 checkpoint_id: 1f1579fb-12a5-6939-bfff-f6b1fbd497b1
 next: ('__start__',)
 state_diff: {}

Step 1
 checkpoint_id: 1f1579fb-12a7-6ef1-8000-3ec3f299bad7
 next: ('set_prompt',)
 state_diff: {}

Step 2
 checkpoint_id: 1f1579fb-12a9-6ce3-8001-4d567bb70f4b
 next: ('identify_bug',)
 state_diff: {'prompt': {'from': None, 'to': 'Bug: Python function returns None unexpectedly'}}

Step 3
 checkpoint_id: 1f1579fb-12ab-6610-8002-2f079f95fc40
 next: ('propose_fix',)
 state_diff: {'bug': {'from': None, 'to': 'Missing return statement in a branch'}}

Step 4
 checkpoint_id: 1f1579fb-12ac-6124-8003-e7f2589dfc40
 next: ('finalize',)
 state_diff: {'fix': {'from': None, 'to': 'Add an explicit return value in every branch'}}

Step 5
 checkpoint_id: 1f1579fb-12ad-6482-8004-b5e562543631
 next: 